# Volti Projesi - Smart Meters in London Veri Seti Hazırlama Pipeline'ı

Bu notebook, **Smart Meters in London** veri seti için bellek kullanımını optimize eden veri temizleme ve feature engineering pipeline'ını uygular.

### Pipeline Adımları:
1. **Metadata Yükleme:** Household classifications (`stdorToU`) ve demographics (`Acorn_grouped`) verilerini yükler ve veri tiplerini optimize eder.
2. **Weather Verisini Yeniden Örnekleme:** Dark Sky saatlik weather kayıtlarını, sayısal özellikler için linear interpolation ve kategorik özellikler için forward-fill kullanarak half-hourly aralıklara dönüştürür.
3. **Consumption Verisini Temizleme ve Eksik Değer Doldurma:** Eksik değerleri (`"Null"` stringleri) işler ve aşırı enerji tüketim değerlerini (yarım saatte **10 kWh** üzeri) sınırlar.
4. **Tarife Etiketleme ve Cost Hesaplama:** Standart flat-rate (**14.228 p/kWh**) ve Dynamic Time-of-Use tarifelerini (High: **67.20p**, Normal: **11.76p**, Low: **3.99p**) uygulayarak her half-hourly kayıt için enerji maliyetini hesaplar.
5. **Feature Engineering:** Zaman serisi için lag feature'ları (örneğin önceki 30 dakika, 24 saat önce ve 1 hafta önceki tüketim) ile datetime tabanlı değişkenler oluşturur.
6. **Batch Export:** Kaggle Kernel **Out-Of-Memory (OOM)** hatalarını önlemek amacıyla verileri optimize edilmiş **Parquet** formatında bloklar halinde sıralı olarak kaydeder.

In [ ]:
import os
import gc
import time
import pandas as pd
import numpy as np

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
# Veri setinin bulunduğu dizinler.
# Kod hem Kaggle ortamında hem de yerel bilgisayarda çalışabilecek şekilde hazırlanmıştır.
KAGGLE_PATH = "/kaggle/input/smart-meters-in-london"
LOCAL_PATH = "c:/Users/ryntm/Desktop/bootcamp/yzta_bootcamp_304/archive"

# Çalışma ortamını otomatik olarak tespit et.
# Eğer Kaggle dizini mevcutsa Kaggle yollarını, değilse yerel yolları kullan.
if os.path.exists(KAGGLE_PATH):
    BASE_DIR = KAGGLE_PATH
    OUTPUT_DIR = "/kaggle/working/processed_data"
else:
    BASE_DIR = LOCAL_PATH
    OUTPUT_DIR = "c:/Users/ryntm/Desktop/bootcamp/yzta_bootcamp_304/processed_data"

# Half-hourly tüketim dosyalarının bulunduğu klasör.
HALF_HOURLY_DIR = os.path.join(BASE_DIR, "halfhourly_dataset", "halfhourly_dataset")

# Çıktı klasörü yoksa oluştur.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Kullanılan dizinleri ekrana yazdırarak doğrulama yap.
print(f"[*] Base directory resolved to: {BASE_DIR}")
print(f"[*] Output directory set to: {OUTPUT_DIR}")

## 1. Load Household Metadata and Resample Weather Data

İlk olarak hanelere ait metadata bilgilerini yüklüyorum ve bellek kullanımını azaltmak için veri tiplerini optimize ediyorum. Daha sonra saatlik hava durumu verilerini, akıllı sayaç verileriyle aynı zaman aralığında çalışabilmek için 30 dakikalık kayıtlara dönüştürüyorum.

In [ ]:
def load_household_info():
    # Hane bilgilerini (tarife tipi ve demografik bilgiler) yükle.
    info_path = os.path.join(BASE_DIR, "informations_households.csv")
    print(f"[*] Loading household metadata from: {info_path}")

    # Sadece kullanılacak sütunları okuyarak gereksiz bellek kullanımını önle.
    hh_info = pd.read_csv(info_path, usecols=["LCLid", "stdorToU", "Acorn_grouped"])

    # Bellek kullanımını azaltmak için kategorik sütunları 'category' tipine dönüştür.
    hh_info["LCLid"] = hh_info["LCLid"].astype("category")
    hh_info["stdorToU"] = hh_info["stdorToU"].astype("category")
    hh_info["Acorn_grouped"] = hh_info["Acorn_grouped"].astype("category")

    return hh_info


In [ ]:
def load_and_resample_weather():
    # Saatlik hava durumu verisini yükle.
    weather_path = os.path.join(BASE_DIR, "weather_hourly_darksky.csv")
    print(f"[*] Loading hourly weather parameters from: {weather_path}")

    weather = pd.read_csv(weather_path)

    # Zaman bilgisini datetime formatına çevir ve indeks olarak ayarla.
    weather["time"] = pd.to_datetime(weather["time"])
    weather = weather.set_index("time").sort_index()

    # Saatlik veriyi akıllı sayaç verileriyle uyumlu olacak şekilde
    # 30 dakikalık zaman aralıklarına dönüştür.
    weather_resampled = weather.resample("30min").asfreq()

    # Sayısal sütunlarda eksik oluşan değerleri lineer interpolasyon ile doldur.
    numeric_cols = weather_resampled.select_dtypes(include=[np.number]).columns
    weather_resampled[numeric_cols] = weather_resampled[numeric_cols].interpolate(method="linear")

    # Kategorik sütunlarda son geçerli değeri kullanarak doldurma işlemi yap.
    categorical_cols = weather_resampled.select_dtypes(exclude=[np.number]).columns
    weather_resampled[categorical_cols] = weather_resampled[categorical_cols].ffill()

    # İndeksi tekrar sütuna çevirerek birleştirme işlemleri için hazır hale getir.
    return weather_resampled.reset_index().rename(columns={"time": "tstp"})

## 2. Block Processing Pipeline Function
Asıl veri hazırlama işlemleri bu fonksiyonda yapılıyor. Her blok tek tek okunuyor, temizleniyor, gerekli özellikler ekleniyor ve model için kullanılabilecek son haline getiriliyor.

In [ ]:
def clean_and_process_block(block_file, hh_info, weather_df):
    block_path = os.path.join(HALF_HOURLY_DIR, block_file)

    if not os.path.exists(block_path):
        print(f"[!] Block file not found: {block_path}")
        return None

    # Bu veri tipini özellikle string okuyorum, içinde "Null" değerler var
    df = pd.read_csv(block_path, dtype={"LCLid": "category", "energy(kWh/hh)": "string"})

    df["tstp"] = pd.to_datetime(df["tstp"])

    # "Null" kayıtlarını temizleyip sayısala çevir
    df["energy(kWh/hh)"] = pd.to_numeric(
        df["energy(kWh/hh)"].str.strip(),
        errors="coerce"
    ).astype("float32")

    # Eksik değer doldurmadan önce sıralamayı garanti edeyim
    df = df.sort_values(["LCLid", "tstp"])

    # Her hane için eksik tüketim değerlerini doldur
    df["energy(kWh/hh)"] = (
        df.groupby("LCLid", observed=False)["energy(kWh/hh)"]
        .ffill()
        .bfill()
    )

    # Ev tüketimi için mantıksız yüksek değerleri sınırla
    df.loc[df["energy(kWh/hh)"] > 10.0, "energy(kWh/hh)"] = 10.0

    # Hane bilgilerini ekle
    df = df.merge(hh_info, on="LCLid", how="left")

    # Önce varsayılan tarifeyi ata
    df["price_pence"] = np.where(
        df["stdorToU"] == "Std",
        14.228,
        11.76
    ).astype("float32")

    is_tou = df["stdorToU"] == "ToU"
    hours = df["tstp"].dt.hour

    # ToU tarifesinde düşük fiyatlı saatler
    df.loc[is_tou & (hours >= 0) & (hours < 7), "price_pence"] = 3.99

    # ToU tarifesinde yoğun saatler
    df.loc[is_tou & (hours >= 16) & (hours < 20), "price_pence"] = 67.20

    # Yarım saatlik maliyeti hesapla
    df["cost_pounds"] = (
        (df["energy(kWh/hh)"] * df["price_pence"]) / 100.0
    ).astype("float32")

    # Hava durumu verisini zaman bilgisine göre ekle
    df = df.merge(weather_df, on="tstp", how="left")

    # Kısa vadeli ve dönemsel tüketim örüntülerini yakalamak için lag'ler
    df["lag_1"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(1).astype("float32")
    df["lag_2"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(2).astype("float32")
    df["lag_48"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(48).astype("float32")
    df["lag_336"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(336).astype("float32")

    # Tarih bilgisinden yeni özellikler üret
    df["hour"] = df["tstp"].dt.hour.astype("int8")
    df["day_of_week"] = df["tstp"].dt.dayofweek.astype("int8")
    df["month"] = df["tstp"].dt.month.astype("int8")

    # Hafta sonlarını ayrıca işaretleyelim
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype("int8")

    # Kategorik alanları da optimize edeyim
    df["LCLid"] = df["LCLid"].astype("category")

    for col in ["precipType", "icon", "summary"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    return df

## 3. Run Pipeline for Selected Blocks
Artık tüm hazırlıklar tamam. Bu adımda seçilen blokları tek tek işleyip çıktılarını Parquet formatında kaydediyorum.

In [ ]:
hh_info = load_household_info()
weather_df = load_and_resample_weather()

# Şimdilik ilk 5 bloğu işliyorum, istersem sayıyı artırabilirim
blocks_to_process = [f"block_{i}.csv" for i in range(5)] 

for block_file in blocks_to_process:
    t_start = time.time()
    # Her bloğu aynı pipeline'dan geçir
    processed = clean_and_process_block(block_file, hh_info, weather_df)
    
    if processed is not None:
        out_name = block_file.replace(".csv", ".parquet")
        out_path = os.path.join(OUTPUT_DIR, out_name)
        # İşlenen bloğu Parquet olarak kaydet, RAM'i yormayalım
        processed.to_parquet(out_path, index=False, compression="snappy")
        print(f"[OK] Processed and exported {block_file} in {time.time() - t_start:.2f}s")
        # Belleği temizleyip bir sonraki bloğa geç
        del processed
        gc.collect()

## 4. Verification Check
Bu son adımda oluşturulan Parquet dosyasını açarak çıktıyı kontrol ediyorum. Dosyanın boyutu, bellek kullanımı ve ilk birkaç satırını inceleyip kontrol edelim. 

In [ ]:
# İlk oluşturulan çıktıyı kontrol etmek için aç
test_path = os.path.join(OUTPUT_DIR, "block_0.parquet")

if os.path.exists(test_path):
    verify_df = pd.read_parquet(test_path)

    print("=== Output File Verification ===")
    print(f"File Shape: {verify_df.shape}")

    # Bellek kullanımını tekrar kontrol edeyim
    print("\nMemory Usage Summary:")
    verify_df.info(memory_usage="deep")

    # İlk birkaç satıra göz at
    print("\nHead Data Preview:")
    display(verify_df.head(10))
else:
    print(f"[!] Verification file not found at: {test_path}")